https://medium.com/@gitaramkanawade/exploring-faiss-chroma-embeddings-inside-vs-code-af23ac68c488

As AI and LLM pipelines evolve, embeddings have become the hidden backbone of search, recommendation, and retrieval systems.
But here’s the challenge: once your embeddings are stored in FAISS or Chroma — they become black boxes.

What text chunks were actually embedded? Are all chunks of the same dimension? We need a visual way to explore embeddings.

The author built a VS Code extension called Vector Explorer designed to open and inspect FAISS and Chroma databases, inside VSCOde.


In [1]:
import os
os.environ["TQDM_DISABLE_WIDGETS"] = "1"  # non garantito su tutte le versioni, da testare

import pandas as pd, matplotlib.pyplot as plt, numpy as np, json, csv
import mne

# sklearn
import sklearn
import sklearn.model_selection

from pypdf import PdfReader
import glob

import kagglehub

from embedding_atlas.widget import EmbeddingAtlasWidget
import time

# split text in chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# embedding and vector store
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from dotenv import load_dotenv
load_dotenv(override=True)  # override=True forza il reload se era già in cache
print(os.getenv("KAGGLE_API_TOKEN") is not None)
print(os.getenv("OPENAI_API_TOKEN") is not None)
print(os.getenv("HF_TOKEN") is not None)

True
True
True


In [2]:
DATA_PATH = '/workspaces/llm_finetuning/FinanceRAG'

if not os.path.exists(DATA_PATH):
    # Download latest version
    path = kagglehub.competition_download('icaif-24-finance-rag-challenge',
                                        output_dir=DATA_PATH)

In [3]:
def open_jsonl(path):
    if os.path.isdir(path):
        path = os.path.join(path, os.listdir(path)[0])
        data = pd.read_json(path, lines=True)
        return data
    with open(path,"r") as file:
            pd.read_json(path, lines=True)
    return data

In [4]:
finder_corpus = open_jsonl(os.path.join(DATA_PATH, 'finder_corpus.jsonl'))
finder_corpus.head(3)

,_id,title,text
0,ADBE20230004,ADBE OVERVIEW,Adobe is a global technology company with a mi...
1,ADBE20230006,ADBE OFFERINGS,"We deliver a wide range of products, services ..."
2,ADBE20230007,ADBE OFFERINGS,"Digital Media. We provide products, services a..."


In [5]:
finder_queries = open_jsonl(os.path.join(DATA_PATH, 'finder_queries.jsonl'))
finder_queries.head(3)

,_id,title,text
0,q00001,,What are the service and product offerings fro...
1,q00002,,MSFT segment breakdown
2,q00003,,Who are Microsoft`s key customers?


In [6]:
# Ground truth
finder_gt = pd.read_csv('/workspaces/llm_finetuning/FinanceRAG/FinDER_qrels.tsv', sep='\t',engine="python", quoting=csv.QUOTE_NONE) 
finder_gt.head(3)

,query_id,corpus_id,score
0,q00001,MSFT20230014,1
1,q00001,MSFT20230015,1
2,q00007,MSFT20231529,1


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " ", ""],  # Paragraph → Line → Sentence → Word → Character
    chunk_size=1000,
    chunk_overlap=0
)

finder_corpus["chunks"] = finder_corpus["text"].apply(lambda t: text_splitter.split_text(t))

In [8]:
texts = []
metadatas = []

for _, row in finder_corpus.iterrows():
    for chunk in row["chunks"]:
        texts.append(chunk)
        metadatas.append({
            "doc_id": row["_id"],
            "title": row["title"]
        })

df = pd.DataFrame({
    "chunk_text": texts,
    "doc_id": [m["doc_id"] for m in metadatas],
    "title": [m["title"] for m in metadatas]
})

In [ ]:
from huggingface_hub import get_safetensors_metadata

model_names = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "BAAI/bge-base-en-v1.5",
    "BAAI/bge-large-en-v1.5",
]

for name in model_names:
    meta = get_safetensors_metadata(name)
    total_params = sum(meta.parameter_count.values())
    print(f"{name}: {total_params/1e6:.1f}M parameters")

sentence-transformers/all-MiniLM-L6-v2: 22.7M parametri
sentence-transformers/all-mpnet-base-v2: 109.5M parametri
BAAI/bge-base-en-v1.5: 109.5M parametri
BAAI/bge-large-en-v1.5: 335.1M parametri


https://huggingface.co/models?library=sentence-transformers&p=4&sort=least_params

In [10]:
embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2',
                                        encode_kwargs={"normalize_embeddings": True},
                                        model_kwargs={"token": os.getenv('HF_TOKEN')},
                                        show_progress=True)

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
embeddings = embedding_model.embed_documents(texts)  

df["embedding"] = list(embeddings)  

Batches:   0%|          | 0/512 [00:00<?, ?it/s]

In [14]:
import inspect
from embedding_atlas.projection import compute_projection

print(inspect.signature(compute_projection))
print(inspect.getsource(compute_projection))

(data_frame: ~IntoDataFrameT, *, inputs: str, modality: str = 'auto', x: str = 'projection_x', y: str = 'projection_y', neighbors: str | None = 'neighbors', embedder: str | collections.abc.Callable | None = None, model: str | None = None, batch_size: int | None = None, max_concurrency: int | None = None, embedder_args: dict | None = None, umap_args: dict | None = None, cache_root: str | pathlib.Path | None = None) -> ~IntoDataFrameT
def compute_projection(
    data_frame: IntoDataFrameT,
    *,
    inputs: str,
    modality: str = "auto",
    x: str = "projection_x",
    y: str = "projection_y",
    neighbors: str | None = "neighbors",
    embedder: str | Callable | None = None,
    model: str | None = None,
    batch_size: int | None = None,
    max_concurrency: int | None = None,
    embedder_args: dict | None = None,
    umap_args: dict | None = None,
    cache_root: str | Path | None = None,
) -> IntoDataFrameT:
    """
    Compute embeddings and generate 2D projections for a DataF

In [17]:
import umap

In [18]:
#  Project embeddings to 2D using UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
projection = reducer.fit_transform(embeddings)

/usr/local/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [19]:
#  Save coordinates into the dataframe
df["projection_x"] = projection[:, 0]
df["projection_y"] = projection[:, 1]

In [20]:
df.to_parquet("/workspaces/llm_finetuning/finder_texts_with_umap.parquet")

In [15]:
from embedding_atlas.projection import async_compute_projection

df = await async_compute_projection(
    df,
    inputs="embedding",
    modality="vector",
    x="projection_x",
    y="projection_y",
    neighbors="neighbors"
)

widget = EmbeddingAtlasWidget(df, text="chunk_text", x="projection_x", y="projection_y")
widget

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))